# 03 Labels: reading the reports with Qwen3-8B

Only 58 of the 4,407 training studies have radiologist labels, so the training targets for the other 4,349 come from their reports. Qwen3-8B, an open multilingual language model, reads every report and answers twelve yes or no questions; the probability it gives to "1" for each answer becomes a soft label.

The model runs offline from Kaggle Models in float16 on two T4 GPUs, in about two hours. The output, `pseudo.csv`, is one of the four label sources that the training notebooks average (notebook 04). This stage first ran as a script; the code below is that script split into cells, unchanged.

Imports, the data path and the twelve finding names. `GOLD_ONLY` switches to the 58 labelled studies for a quick check of a prompt; `TOK` caps the tokens per batch so that a batch fits in GPU memory.

In [1]:
import glob, os, re, time
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
from transformers import AutoModelForCausalLM, AutoTokenizer

R = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
GOLD_ONLY = False
TOK = 13000
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",
       "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

The instructions. Each finding gets a one-line definition that settles the usual ambiguities: a sprain counts as an ACL or MCL injury, meniscal degeneration without a tear does not count, and cartilage loss counts as osteoarthritis of its compartment. The answer format is fixed to twelve lines of `Name: 0` or `Name: 1`, and the model's thinking mode is off.

In [2]:
SYSTEM = """You read a knee MRI radiology report (any language) and label 12 findings, 1 = present, 0 = absent or not mentioned.
ACL: anterior cruciate ligament tear, rupture or sprain (partial or complete); not mere mild degeneration.
MCL: medial collateral ligament tear or sprain of any grade.
Medial Meniscus / Lateral Meniscus: tear of that meniscus (includes radial, root, bucket handle, complex, horizontal, displaced flap); ignore intact meniscus, degeneration only or grade 1-2 signal without tear.
Medial OA: osteoarthritis or cartilage loss / chondropathy / chondrosis / osteophytes in the medial femorotibial compartment.
Lateral OA: same for the lateral femorotibial compartment.
PF OA: chondropathy, cartilage defect or osteoarthritis of the patellofemoral joint (patella or trochlea).
Effusion: joint effusion, hydrops, excess joint fluid.
Synovitis: synovitis, synovial thickening or proliferation.
Baker's: popliteal or Baker's cyst.
Contusion: bone contusion, bone bruise, marrow or bone edema.
Fracture: fracture, insufficiency fracture, osteochondral or impaction fracture.
Answer with exactly 12 lines in this order, each 'Name: 0' or 'Name: 1', nothing else."""


def prompt(tok, rep):
    m = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": str(rep)[:5000]}]
    return tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False)

The labelling run. Reports are cut to 3,500 characters, sorted by length and grouped into batches of up to 24 within the token budget, which keeps padding small. Generation is greedy. For each of the twelve answer digits the code keeps the probability of "1" against "0", so a confident no and a hesitant no get different labels. A report whose answer does not contain exactly twelve digits is left empty. At the end the labels are scored against the 58 radiologist-labelled studies.

In [3]:
def main():
    tr = pd.read_csv(f"{R}/train.csv", encoding_errors="replace")
    if GOLD_ONLY:
        tr = tr[tr[LAB[0]].notna()].reset_index(drop=True)
    path = os.path.dirname(glob.glob("/kaggle/input/models/**/config.json", recursive=True)[0])
    print(path)
    tok = AutoTokenizer.from_pretrained(path, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16, device_map="auto", max_memory={0: "8.5GiB", 1: "9GiB"}).eval()
    ids = [tok.encode(str(x)) for x in tr.Report]
    order = np.argsort([len(i) for i in ids])
    d0, d1 = tok.encode("0")[0], tok.encode("1")[0]
    out = np.full((len(tr), 12), np.nan, np.float32)
    t0 = time.time()
    batches, cur = [], []
    for i in order:
        if cur and (len(cur) + 1) * (min(len(ids[i]), 1800) + 500) > TOK or len(cur) >= 24:
            batches.append(cur)
            cur = []
        cur.append(i)
    batches.append(cur)
    for b, idx in enumerate(batches):
        enc = tok([prompt(tok, tr.Report.iloc[i][:3500]) for i in idx], return_tensors="pt", padding=True).to("cuda:0")
        with torch.no_grad():
            g = model.generate(**enc, max_new_tokens=90, do_sample=False, output_scores=True, return_dict_in_generate=True)
        seq, sc = g.sequences[:, enc.input_ids.shape[1]:], torch.stack(g.scores, 1).float()
        for r, i in enumerate(idx):
            probs, toks = [], seq[r].tolist()
            for t, tk in enumerate(toks):
                if tk in (d0, d1):
                    p = torch.softmax(sc[r, t, [d0, d1]], 0)[1].item()
                    probs.append(p)
            if len(probs) == 12:
                out[i] = probs
        if b % 10 == 0:
            print(b, len(batches), round(time.time() - t0), flush=True)
    res = pd.DataFrame(out, columns=LAB)
    res.insert(0, "StudyInstanceUID", tr.StudyInstanceUID)
    res.to_csv("pseudo.csv", index=False)
    print("parsed", np.isfinite(out[:, 0]).mean())
    g = tr[tr[LAB[0]].notna()]
    gi = g.index.values
    for k, c in enumerate(LAB):
        y, p = g[c].values, out[gi, k]
        m = np.isfinite(p)
        auc = roc_auc_score(y[m], p[m]) if len(set(y[m])) > 1 else float("nan")
        print(f"{c:18s} auc {auc:.3f} acc {((p[m] > .5) == y[m]).mean():.3f}")

Run it on all 4,407 reports. Progress is printed every ten batches with the elapsed seconds.

In [4]:
main()

/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1


`torch_dtype` is deprecated! Use `dtype` instead!


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


0 303 21
10 303 250
20 303 488
30 303 716
40 303 945
50 303 1176
60 303 1409
70 303 1643
80 303 1877
90 303 2109
100 303 2335
110 303 2562
120 303 2786
130 303 3012
140 303 3241
150 303 3467
160 303 3696
170 303 3929
180 303 4156
190 303 4387
200 303 4623
210 303 4853
220 303 5084
230 303 5321
240 303 5552
250 303 5785
260 303 6025
270 303 6257
280 303 6494
290 303 6738
300 303 6987
parsed 0.9977308826866349
ACL                auc 0.884 acc 0.776
MCL                auc 0.912 acc 0.810
Medial Meniscus    auc 0.830 acc 0.810
Lateral Meniscus   auc 0.867 acc 0.793
Medial OA          auc 0.942 acc 0.897
Lateral OA         auc 0.832 acc 0.845
PF OA              auc 0.868 acc 0.845
Effusion           auc 0.734 acc 0.724
Synovitis          auc 0.810 acc 0.793
Baker's            auc 0.879 acc 0.914
Contusion          auc 0.858 acc 0.759
Fracture           auc 0.869 acc 0.810


**Conclusion.** 303 batches ran in about 115 minutes, and 99.8% of the reports gave exactly twelve answers. Against the radiologist labels the probabilities score from 0.734 (effusion) to 0.942 (medial OA), 0.857 on average, below the best public label table (0.893). Averaged with three public label tables they reach 0.897, which is why the training notebooks use the average (notebook 04).